<a href="https://colab.research.google.com/github/MAHMOUD-lab26/Image-Processing-and-Computer-Vision/blob/main/ComputerVisionProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# University Computer Vision Project: Pose Estimation for Exercise Analysis

This notebook provides a complete solution for 2D pose estimation, exercise repetition counting (squats and push-ups), and form quality assessment using `YOLOv8s-pose` and `OpenCV`. The analysis includes Exponential Moving Average (EMA) smoothing for keypoints and generates a structured JSON output summarizing the performance.

## 1. Environment Setup and Library Installation

In [7]:
# Install necessary libraries
%pip install ultralytics opencv-python numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.4 MB/s eta 0:00:00


In [8]:
# Import required libraries
import cv2
import numpy as np
from ultralytics import YOLO
import json
import uuid
import math
from google.colab import files
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 2. Load YOLOv8s-pose Model

In [9]:
# Load the YOLOv8s-pose model
# This model is pre-trained to detect 17 keypoints on the human body.
model = YOLO('yolov8s-pose.pt')

## 3. Video Upload and Processing Setup

This section allows you to upload your MP4 video files directly to the Colab environment for analysis. Make sure to upload both the 'Squats.mp4' and 'PushUps.mp4' files as mentioned.

In [10]:
import os

# Function to find video files directly from the /content/ directory
def find_video_files(directory='.'):
    video_list = []
    for filename in os.listdir(directory):
        if filename.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
            video_list.append(os.path.join(directory, filename))
    return video_list

print("Searching for video files in the current directory...")
video_files_on_disk = find_video_files('/content/')
print("Found video files:", video_files_on_disk)

# Identify squats and pushups videos based on keywords in their filenames
squats_video_path = None
pushups_video_path = None

for video_path in video_files_on_disk:
    if "squats" in os.path.basename(video_path).lower():
        # Prefer files with 'squats' and '.mp4' if multiple exist
        if squats_video_path is None or '.mp4' in os.path.basename(video_path).lower():
            squats_video_path = video_path
    elif "pushups" in os.path.basename(video_path).lower():
        if pushups_video_path is None or '.mp4' in os.path.basename(video_path).lower():
            pushups_video_path = video_path

if squats_video_path:
    print(f"Squats video identified: {squats_video_path}")
else:
    print("Squats video not found. Please ensure a video with 'squats' in its name is in the /content/ directory.")

if pushups_video_path:
    print(f"Push-ups video identified: {pushups_video_path}")
else:
    print("Push-ups video not found. Please ensure a video with 'pushups' in its name is in the /content/ directory.")


Searching for video files in the current directory...
Found video files: ['/content/PushUps.mp4', '/content/Squats.mp4']
Squats video identified: /content/Squats.mp4
Push-ups video identified: /content/PushUps.mp4


## 4. Helper Functions for Pose Analysis

This section defines all the utility functions required for processing keypoints, calculating angles, smoothing data, and detecting exercise repetitions along with form assessment.

In [11]:
def calculate_angle(keypoints, p1_idx, p2_idx, p3_idx):
    """
    Calculates the angle (in degrees) between three keypoints.
    p2_idx is the vertex of the angle.
    """
    # Ensure keypoints are valid (x, y, confidence) format and points exist
    if p1_idx >= len(keypoints) or p2_idx >= len(keypoints) or p3_idx >= len(keypoints):
        return None

    # Extract coordinates
    p1 = np.array([keypoints[p1_idx][0], keypoints[p1_idx][1]])
    p2 = np.array([keypoints[p2_idx][0], keypoints[p2_idx][1]])
    p3 = np.array([keypoints[p3_idx][0], keypoints[p3_idx][1]])

    # Calculate vectors
    v1 = p1 - p2
    v2 = p3 - p2

    # Calculate dot product and magnitudes
    dot_product = np.dot(v1, v2)
    magnitude_v1 = np.linalg.norm(v1)
    magnitude_v2 = np.linalg.norm(v2)

    if magnitude_v1 == 0 or magnitude_v2 == 0:
        return None # Avoid division by zero

    # Calculate cosine of the angle
    cosine_angle = dot_product / (magnitude_v1 * magnitude_v2)
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0) # Clip to avoid floating point errors

    # Calculate angle in radians and then convert to degrees
    angle_rad = np.arccos(cosine_angle)
    angle_deg = np.degrees(angle_rad)

    return angle_deg

def ema_smoothing(data, alpha=0.2):
    """
    Applies Exponential Moving Average (EMA) smoothing to a list of data points.
    `data` can be a list of scalar values or lists/arrays (e.g., [x, y] keypoints).
    `alpha` is the smoothing factor (0 < alpha <= 1).
    """
    if not data:
        return []

    smoothed_data = [data[0]] # Initialize with the first data point
    for i in range(1, len(data)):
        if isinstance(data[i], (list, np.ndarray)):
            # For lists/arrays (like keypoints)
            smoothed_value = [alpha * current + (1 - alpha) * prev for current, prev in zip(data[i], smoothed_data[-1])]
        else:
            # For scalar values
            smoothed_value = alpha * data[i] + (1 - alpha) * smoothed_data[-1]
        smoothed_data.append(smoothed_value)

    return smoothed_data

# --- Squat Repetition Detection and Form Assessment ---
# Keypoint indices for squats:
# 0: nose, 1: left_eye, 2: right_eye, 3: left_ear, 4: right_ear,
# 5: left_shoulder, 6: right_shoulder, 7: left_elbow, 8: right_elbow,
# 9: left_wrist, 10: right_wrist, 11: left_hip, 12: right_hip,
# 13: left_knee, 14: right_knee, 15: left_ankle, 16: right_ankle

SQUAT_HIP_KNEE_THRESHOLD_DOWN = 90 # Angle (degrees) when hip-knee is considered 'down' (e.g., squat depth)
SQUAT_HIP_KNEE_THRESHOLD_UP = 160  # Angle (degrees) when hip-knee is considered 'up' (e.g., standing straight)
SQUAT_MIN_REP_ANGLE_CHANGE = 30   # Minimum angle change to count as a rep

def is_squat_rep(prev_hip_knee_angle, current_hip_knee_angle, squat_state):
    """
    Detects a squat repetition based on hip-knee angle changes.
    Returns (is_rep_counted, new_squat_state)
    """
    is_rep = False
    new_squat_state = squat_state

    if prev_hip_knee_angle is None or current_hip_knee_angle is None:
        return False, squat_state

    # State 0: Initial/Upright
    # State 1: Descending/Squatting
    # State 2: Ascending/Coming up

    if squat_state == 0: # Initial/Upright state
        if current_hip_knee_angle < SQUAT_HIP_KNEE_THRESHOLD_DOWN and \
           abs(current_hip_knee_angle - prev_hip_knee_angle) > SQUAT_MIN_REP_ANGLE_CHANGE:
            new_squat_state = 1 # Started squatting down
    elif squat_state == 1: # Descending/Squatting state
        if current_hip_knee_angle > SQUAT_HIP_KNEE_THRESHOLD_UP and \
           abs(current_hip_knee_angle - prev_hip_knee_angle) > SQUAT_MIN_REP_ANGLE_CHANGE:
            is_rep = True # Completed a full rep (down and up)
            new_squat_state = 0 # Reset to upright

    return is_rep, new_squat_state

def assess_squat_form(hip_knee_angle):
    """
    Assesses squat form based on hip-knee angle at the deepest point.
    """
    if hip_knee_angle is None:
        return "Bad Form (no angle data)"

    # A deeper squat typically has a smaller hip-knee angle
    if hip_knee_angle < SQUAT_HIP_KNEE_THRESHOLD_DOWN: # e.g., below 90 degrees
        return "Good Form"
    else:
        return "Bad Form (insufficient depth)"


# --- Push-up Repetition Detection and Form Assessment ---
# Keypoint indices for push-ups:
# Elbow angle: Shoulder (5/6) - Elbow (7/8) - Wrist (9/10)
# Body alignment: Shoulder (5/6) - Hip (11/12) - Ankle (15/16)

PUSHUP_ELBOW_ANGLE_DOWN_THRESHOLD = 90 # Angle when elbow is bent (down position)
PUSHUP_ELBOW_ANGLE_UP_THRESHOLD = 160  # Angle when elbow is extended (up position)
PUSHUP_BODY_ALIGNMENT_THRESHOLD = 170 # Angle for good body alignment (straight line)
PUSHUP_MIN_REP_ANGLE_CHANGE = 20    # Minimum angle change to count as a rep

def is_pushup_rep(prev_elbow_angle, current_elbow_angle, pushup_state):
    """
    Detects a push-up repetition based on elbow angle changes.
    Returns (is_rep_counted, new_pushup_state)
    """
    is_rep = False
    new_pushup_state = pushup_state

    if prev_elbow_angle is None or current_elbow_angle is None:
        return False, pushup_state

    # State 0: Initial/Up position (arms extended)
    # State 1: Descending/Down position (elbows bent)

    if pushup_state == 0: # Initial/Up position
        if current_elbow_angle < PUSHUP_ELBOW_ANGLE_DOWN_THRESHOLD and \
           abs(current_elbow_angle - prev_elbow_angle) > PUSHUP_MIN_REP_ANGLE_CHANGE:
            new_pushup_state = 1 # Started moving down
    elif pushup_state == 1: # Descending/Down position
        if current_elbow_angle > PUSHUP_ELBOW_ANGLE_UP_THRESHOLD and \
           abs(current_elbow_angle - prev_elbow_angle) > PUSHUP_MIN_REP_ANGLE_CHANGE:
            is_rep = True # Completed a full rep (down and up)
            new_pushup_state = 0 # Reset to up position

    return is_rep, new_pushup_state

def assess_pushup_form(shoulder_hip_ankle_angle):
    """
    Assesses push-up form based on body alignment (shoulder-hip-ankle angle).
    """
    if shoulder_hip_ankle_angle is None:
        return "Bad Form (no angle data)"

    # A straight body forms an angle close to 180 degrees
    if shoulder_hip_ankle_angle > PUSHUP_BODY_ALIGNMENT_THRESHOLD: # e.g., above 170 degrees
        return "Good Form"
    else:
        return "Bad Form (poor body alignment)"

## 5. Main Video Processing Function

This function takes a video file, processes it frame by frame, applies pose estimation, smoothes keypoints using EMA, detects repetitions for either squats or push-ups, and assesses form quality. It stores the results for later summary.

In [12]:
def process_video(video_path, exercise_type):
    """
    Processes a video for pose estimation, rep counting, and form assessment.

    Args:
        video_path (str): The path to the video file.
        exercise_type (str): 'squats' or 'pushups'.

    Returns:
        dict: A dictionary containing total reps, good form reps, and detailed rep data.
    """
    print(f"\nProcessing {exercise_type} video: {video_path}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return None

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Initialize state variables for rep counting and form assessment
    total_reps = 0
    good_form_reps = 0
    all_keypoints = [] # To store keypoints for EMA smoothing

    # Exercise-specific variables
    if exercise_type == 'squats':
        rep_state = 0 # 0: upright, 1: descending, 2: ascending (for squats, we just need up/down)
        prev_hip_knee_angle = None
        deepest_squat_angle_in_rep = None
        rep_data = []
    elif exercise_type == 'pushups':
        rep_state = 0 # 0: up, 1: down
        prev_elbow_angle = None
        body_alignment_angle_in_rep = None
        rep_data = []
    else:
        print("Error: Unknown exercise type.")
        return None

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Perform pose estimation
        results = model(frame, verbose=False) # verbose=False to suppress output for each frame

        # Extract keypoints
        if results[0].keypoints.data.numel() > 0: # Check if keypoints are detected
            # Assuming single person detection, take the first detection
            keypoints = results[0].keypoints.data[0].cpu().numpy() # x, y, confidence
            all_keypoints.append(keypoints)
        else:
            # If no keypoints detected, append None or the last known keypoints
            all_keypoints.append(None)

        frame_idx += 1

    cap.release()

    if not all_keypoints:
        print(f"No keypoints detected in {video_path}.")
        return {"total_reps": 0, "good_form_reps": 0, "reps": []}

    # Apply EMA smoothing to all detected keypoints after processing all frames
    # Filter out None values before smoothing, and re-insert them or use interpolation if needed
    # For simplicity here, we'll process available keypoints and ignore None for smoothing input
    # A more robust solution might interpolate None frames or use a different smoothing approach

    # For EMA, we need to smooth each keypoint's X and Y coordinate independently over time.
    # Let's restructure keypoints for smoothing: [ [kp1_x, kp1_y], [kp2_x, kp2_y], ... ] for each frame
    smoothed_keypoints_per_frame = []
    valid_frames_keypoints = [kp for kp in all_keypoints if kp is not None]

    if valid_frames_keypoints:
        # Reshape for easier EMA: (num_frames, num_keypoints, 3)
        # Convert to list of lists for EMA function: [frame1_kp_list, frame2_kp_list, ...]
        # where frame_kp_list is [ [x1,y1], [x2,y2], ... ]
        reshaped_keypoints = []
        for frame_kp in valid_frames_keypoints:
            reshaped_keypoints.append([[kp[0], kp[1]] for kp in frame_kp])

        # Apply EMA smoothing per keypoint over all valid frames
        # This requires transposing: list of (frames, [x,y]) for each keypoint
        if reshaped_keypoints:
            num_kps = len(reshaped_keypoints[0])
            smoothed_kp_coords = [[] for _ in range(num_kps)] # List of lists, one for each keypoint

            for kp_idx in range(num_kps):
                kp_time_series_x = [frame_kp[kp_idx][0] for frame_kp in reshaped_keypoints]
                kp_time_series_y = [frame_kp[kp_idx][1] for frame_kp in reshaped_keypoints]
                smoothed_x = ema_smoothing(kp_time_series_x, alpha=0.3)
                smoothed_y = ema_smoothing(kp_time_series_y, alpha=0.3)
                smoothed_kp_coords[kp_idx] = [[sx, sy] for sx, sy in zip(smoothed_x, smoothed_y)]

            # Reconstruct smoothed keypoints back to (num_frames, num_keypoints, 2) structure
            # smoothed_keypoints_per_frame will have only valid frames
            smoothed_keypoints_per_frame = []
            for i in range(len(smoothed_kp_coords[0])):
                frame_smoothed_kps = []
                for kp_idx in range(num_kps):
                    frame_smoothed_kps.append(smoothed_kp_coords[kp_idx][i] + [1.0]) # Add dummy confidence for structure
                smoothed_keypoints_per_frame.append(np.array(frame_smoothed_kps))

    else:
        return {"total_reps": 0, "good_form_reps": 0, "reps": []}


    # Re-process frames with smoothed keypoints for rep counting and form assessment
    valid_frame_idx = 0
    for i, keypoints_raw in enumerate(all_keypoints):
        if keypoints_raw is None:
            continue # Skip frames where no person was detected initially

        keypoints = smoothed_keypoints_per_frame[valid_frame_idx]
        valid_frame_idx += 1

        # --- Squats --- (Left Hip: 11, Left Knee: 13, Left Ankle: 15)
        # Or Right Hip: 12, Right Knee: 14, Right Ankle: 16
        if exercise_type == 'squats':
            # Using left side for angles
            hip_idx, knee_idx, ankle_idx = 11, 13, 15
            current_hip_knee_angle = calculate_angle(keypoints, hip_idx, knee_idx, ankle_idx)

            if current_hip_knee_angle is not None:
                # Update deepest squat angle if descending
                if rep_state == 1: # If descending
                    if deepest_squat_angle_in_rep is None or current_hip_knee_angle < deepest_squat_angle_in_rep:
                        deepest_squat_angle_in_rep = current_hip_knee_angle

                is_rep, rep_state = is_squat_rep(prev_hip_knee_angle, current_hip_knee_angle, rep_state)

                if is_rep:
                    total_reps += 1
                    form_quality = assess_squat_form(deepest_squat_angle_in_rep)
                    if form_quality == "Good Form":
                        good_form_reps += 1
                    rep_data.append({"rep_number": total_reps, "form_quality": form_quality})
                    deepest_squat_angle_in_rep = None # Reset for next rep

                prev_hip_knee_angle = current_hip_knee_angle

        # --- Push-ups --- (Left Shoulder: 5, Left Elbow: 7, Left Wrist: 9)
        # Body alignment: Left Shoulder: 5, Left Hip: 11, Left Ankle: 15
        elif exercise_type == 'pushups':
            # Using left side for angles
            shoulder_idx, elbow_idx, wrist_idx = 5, 7, 9
            current_elbow_angle = calculate_angle(keypoints, shoulder_idx, elbow_idx, wrist_idx)

            # For body alignment
            hip_idx = 11
            ankle_idx = 15
            current_body_alignment_angle = calculate_angle(keypoints, shoulder_idx, hip_idx, ankle_idx)

            if current_elbow_angle is not None and current_body_alignment_angle is not None:
                # Keep track of body alignment angle at 'down' position
                if rep_state == 1: # If descending (elbows bending)
                    if body_alignment_angle_in_rep is None or current_body_alignment_angle > body_alignment_angle_in_rep:
                         body_alignment_angle_in_rep = current_body_alignment_angle

                is_rep, rep_state = is_pushup_rep(prev_elbow_angle, current_elbow_angle, rep_state)

                if is_rep:
                    total_reps += 1
                    form_quality = assess_pushup_form(body_alignment_angle_in_rep)
                    if form_quality == "Good Form":
                        good_form_reps += 1
                    rep_data.append({"rep_number": total_reps, "form_quality": form_quality})
                    body_alignment_angle_in_rep = None # Reset for next rep

                prev_elbow_angle = current_elbow_angle

    print(f"Finished processing {exercise_type}: Total Reps = {total_reps}, Good Form Reps = {good_form_reps}")
    return {"total_reps": total_reps, "good_form_reps": good_form_reps, "reps": rep_data}

# --- Process Videos and Generate Output ---


This section runs the `process_video` function for the uploaded squat and push-up videos (if available). It then compiles the results into the specified JSON format and saves it to a file.

## 5. Main Video Processing Function

This function takes a video file, processes it frame by frame, applies pose estimation, smoothes keypoints using EMA, detects repetitions for either squats or push-ups, and assesses form quality. It stores the results for later summary.

In [13]:
def process_video(video_path, exercise_type):
    """
    Processes a video for pose estimation, rep counting, and form assessment.

    Args:
        video_path (str): The path to the video file.
        exercise_type (str): 'squats' or 'pushups'.

    Returns:
        dict: A dictionary containing total reps, good form reps, and detailed rep data.
    """
    print(f"\nProcessing {exercise_type} video: {video_path}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return None

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Initialize state variables for rep counting and form assessment
    total_reps = 0
    good_form_reps = 0
    all_keypoints = [] # To store keypoints for EMA smoothing

    # Exercise-specific variables
    if exercise_type == 'squats':
        rep_state = 0 # 0: upright, 1: descending, 2: ascending (for squats, we just need up/down)
        prev_hip_knee_angle = None
        deepest_squat_angle_in_rep = None
        rep_data = []
    elif exercise_type == 'pushups':
        rep_state = 0 # 0: up, 1: down
        prev_elbow_angle = None
        body_alignment_angle_in_rep = None
        rep_data = []
    else:
        print("Error: Unknown exercise type.")
        return None

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Perform pose estimation
        results = model(frame, verbose=False) # verbose=False to suppress output for each frame

        # Extract keypoints
        if results[0].keypoints.data.numel() > 0: # Check if keypoints are detected
            # Assuming single person detection, take the first detection
            keypoints = results[0].keypoints.data[0].cpu().numpy() # x, y, confidence
            all_keypoints.append(keypoints)
        else:
            # If no keypoints detected, append None or the last known keypoints
            all_keypoints.append(None)

        frame_idx += 1

    cap.release()

    if not all_keypoints:
        print(f"No keypoints detected in {video_path}.")
        return {"total_reps": 0, "good_form_reps": 0, "reps": []}

    # Apply EMA smoothing to all detected keypoints after processing all frames
    # Filter out None values before smoothing, and re-insert them or use interpolation if needed
    # For simplicity here, we'll process available keypoints and ignore None for smoothing input
    # A more robust solution might interpolate None frames or use a different smoothing approach

    # For EMA, we need to smooth each keypoint's X and Y coordinate independently over time.
    # Let's restructure keypoints for smoothing: [ [kp1_x, kp1_y], [kp2_x, kp2_y], ... ] for each frame
    smoothed_keypoints_per_frame = []
    valid_frames_keypoints = [kp for kp in all_keypoints if kp is not None]

    if valid_frames_keypoints:
        # Reshape for easier EMA: (num_frames, num_keypoints, 3)
        # Convert to list of lists for EMA function: [frame1_kp_list, frame2_kp_list, ...]
        # where frame_kp_list is [ [x1,y1], [x2,y2], ... ]
        reshaped_keypoints = []
        for frame_kp in valid_frames_keypoints:
            reshaped_keypoints.append([[kp[0], kp[1]] for kp in frame_kp])

        # Apply EMA smoothing per keypoint over all valid frames
        # This requires transposing: list of (frames, [x,y]) for each keypoint
        if reshaped_keypoints:
            num_kps = len(reshaped_keypoints[0])
            smoothed_kp_coords = [[] for _ in range(num_kps)] # List of lists, one for each keypoint

            for kp_idx in range(num_kps):
                kp_time_series_x = [frame_kp[kp_idx][0] for frame_kp in reshaped_keypoints]
                kp_time_series_y = [frame_kp[kp_idx][1] for frame_kp in reshaped_keypoints]
                smoothed_x = ema_smoothing(kp_time_series_x, alpha=0.3)
                smoothed_y = ema_smoothing(kp_time_series_y, alpha=0.3)
                smoothed_kp_coords[kp_idx] = [[sx, sy] for sx, sy in zip(smoothed_x, smoothed_y)]

            # Reconstruct smoothed keypoints back to (num_frames, num_keypoints, 2) structure
            # smoothed_keypoints_per_frame will have only valid frames
            smoothed_keypoints_per_frame = []
            for i in range(len(smoothed_kp_coords[0])):
                frame_smoothed_kps = []
                for kp_idx in range(num_kps):
                    frame_smoothed_kps.append(smoothed_kp_coords[kp_idx][i] + [1.0]) # Add dummy confidence for structure
                smoothed_keypoints_per_frame.append(np.array(frame_smoothed_kps))

    else:
        return {"total_reps": 0, "good_form_reps": 0, "reps": []}


    # Re-process frames with smoothed keypoints for rep counting and form assessment
    valid_frame_idx = 0
    for i, keypoints_raw in enumerate(all_keypoints):
        if keypoints_raw is None:
            continue # Skip frames where no person was detected initially

        keypoints = smoothed_keypoints_per_frame[valid_frame_idx]
        valid_frame_idx += 1

        # --- Squats --- (Left Hip: 11, Left Knee: 13, Left Ankle: 15)
        # Or Right Hip: 12, Right Knee: 14, Right Ankle: 16
        if exercise_type == 'squats':
            # Using left side for angles
            hip_idx, knee_idx, ankle_idx = 11, 13, 15
            current_hip_knee_angle = calculate_angle(keypoints, hip_idx, knee_idx, ankle_idx)

            if current_hip_knee_angle is not None:
                # Update deepest squat angle if descending
                if rep_state == 1: # If descending
                    if deepest_squat_angle_in_rep is None or current_hip_knee_angle < deepest_squat_angle_in_rep:
                        deepest_squat_angle_in_rep = current_hip_knee_angle

                is_rep, rep_state = is_squat_rep(prev_hip_knee_angle, current_hip_knee_angle, rep_state)

                if is_rep:
                    total_reps += 1
                    form_quality = assess_squat_form(deepest_squat_angle_in_rep)
                    if form_quality == "Good Form":
                        good_form_reps += 1
                    rep_data.append({"rep_number": total_reps, "form_quality": form_quality})
                    deepest_squat_angle_in_rep = None # Reset for next rep

                prev_hip_knee_angle = current_hip_knee_angle

        # --- Push-ups --- (Left Shoulder: 5, Left Elbow: 7, Left Wrist: 9)
        # Body alignment: Left Shoulder: 5, Left Hip: 11, Left Ankle: 15
        elif exercise_type == 'pushups':
            # Using left side for angles
            shoulder_idx, elbow_idx, wrist_idx = 5, 7, 9
            current_elbow_angle = calculate_angle(keypoints, shoulder_idx, elbow_idx, wrist_idx)

            # For body alignment
            hip_idx = 11
            ankle_idx = 15
            current_body_alignment_angle = calculate_angle(keypoints, shoulder_idx, hip_idx, ankle_idx)

            if current_elbow_angle is not None and current_body_alignment_angle is not None:
                # Keep track of body alignment angle at 'down' position
                if rep_state == 1: # If descending (elbows bending)
                    if body_alignment_angle_in_rep is None or current_body_alignment_angle > body_alignment_angle_in_rep:
                         body_alignment_angle_in_rep = current_body_alignment_angle

                is_rep, rep_state = is_pushup_rep(prev_elbow_angle, current_elbow_angle, rep_state)

                if is_rep:
                    total_reps += 1
                    form_quality = assess_pushup_form(body_alignment_angle_in_rep)
                    if form_quality == "Good Form":
                        good_form_reps += 1
                    rep_data.append({"rep_number": total_reps, "form_quality": form_quality})
                    body_alignment_angle_in_rep = None # Reset for next rep

                prev_elbow_angle = current_elbow_angle

    print(f"Finished processing {exercise_type}: Total Reps = {total_reps}, Good Form Reps = {good_form_reps}")
    return {"total_reps": total_reps, "good_form_reps": good_form_reps, "reps": rep_data}

# --- Process Videos and Generate Output ---


## 6. Execute Analysis and Generate JSON Output

This section runs the `process_video` function for the uploaded squat and push-up videos (if available). It then compiles the results into the specified JSON format and saves it to a file.

In [14]:
def process_video(video_path, exercise_type):
    """
    Processes a video for pose estimation, rep counting, and form assessment.
    Uses ultra-high-sensitivity calibration to capture shallow or fast reps.
    """
    print(f"\nProcessing {exercise_type} video: {video_path}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return None

    all_raw_angles = []

    # First Pass: Calibration & Keypoint Extraction
    while True:
        ret, frame = cap.read()
        if not ret: break

        results = model(frame, verbose=False)
        if results[0].keypoints.data.numel() > 0:
            kp = results[0].keypoints.data[0].cpu().numpy()
            if exercise_type == 'squats':
                angle = calculate_angle(kp, 11, 13, 15) # Hip-Knee-Ankle
            else:
                angle = calculate_angle(kp, 5, 7, 9)   # Shoulder-Elbow-Wrist
            if angle: all_raw_angles.append(angle)
    cap.release()

    if not all_raw_angles:
        print(f"No poses detected in {video_path}.")
        return {"total_reps": 0, "good_form_reps": 0, "reps": []}

    # Ultra-High-Sensitivity Auto-Calibration
    min_a = min(all_raw_angles)
    max_a = max(all_raw_angles)
    range_a = max_a - min_a

    # Further increased sensitivity for pushups
    # Down: Triggered at 10% range from the top
    # Up: Triggered at 15% range from the top
    if exercise_type == 'pushups':
        dynamic_down = max_a - (range_a * 0.21)
        dynamic_up = max_a - (range_a * 0.21)
    else:
        dynamic_down = max_a - (range_a * 0.25)
        dynamic_up = max_a - (range_a * 0.25)

    print(f"Calibration: Absolute range {min_a:.1f}° to {max_a:.1f}°")
    print(f"Ultra-Sensitivity Thresholds: Down < {dynamic_down:.1f}°, Up > {dynamic_up:.1f}°")

    total_reps = 0
    good_form_reps = 0
    rep_state = 0 # 0: Up, 1: Down
    deepest_angle_in_rep = None
    rep_data = []

    # EMA Smoothing for logic (alpha=0.3)
    smoothed_angles = ema_smoothing(all_raw_angles, alpha=0.3)

    for current_angle in smoothed_angles:
        # Minimal buffer (1 degree) to prevent missing small movements
        if rep_state == 0 and current_angle < (dynamic_down - 1):
            rep_state = 1
            deepest_angle_in_rep = current_angle
        elif rep_state == 1:
            if deepest_angle_in_rep is None or current_angle < deepest_angle_in_rep:
                deepest_angle_in_rep = current_angle

            if current_angle > dynamic_up:
                total_reps += 1
                # Form standard: Squat < 110 deg, Pushup < 100 deg
                if exercise_type == 'squats':
                    is_good = "Good Form" if deepest_angle_in_rep < 110 else "Bad Form (Depth)"
                else: # Pushups
                    is_good = "Good Form" if deepest_angle_in_rep < 100 else "Bad Form (Depth)"

                if is_good == "Good Form":
                    good_form_reps += 1

                rep_data.append({"rep": total_reps, "quality": is_good})
                rep_state = 0
                deepest_angle_in_rep = None

    print(f"Results: {total_reps} reps counted.")
    return {"total_reps": total_reps, "good_form_reps": good_form_reps, "reps": rep_data}

In [15]:
analysis_results = {
    "video_id": str(uuid.uuid4()),
    "summary": {
        "squats": {"total_reps": 0, "good_form_reps": 0},
        "pushups": {"total_reps": 0, "good_form_reps": 0}
    }
}

# Run analysis for Squats
if squats_video_path:
    print("--- Analyzing Squats ---")
    squats_data = process_video(squats_video_path, 'squats')
    if squats_data:
        analysis_results["summary"]["squats"]["total_reps"] = squats_data["total_reps"]
        analysis_results["summary"]["squats"]["good_form_reps"] = squats_data["good_form_reps"]

# Run analysis for Pushups
if pushups_video_path:
    print("\n--- Analyzing Pushups ---")
    pushups_data = process_video(pushups_video_path, 'pushups')
    if pushups_data:
        analysis_results["summary"]["pushups"]["total_reps"] = pushups_data["total_reps"]
        analysis_results["summary"]["pushups"]["good_form_reps"] = pushups_data["good_form_reps"]

# Finalize and Save Output
output_filename = "exercise_analysis_results.json"
with open(output_filename, 'w') as f:
    json.dump(analysis_results, f, indent=2)

print(f"\nAnalysis complete. Final JSON Result:")
print(json.dumps(analysis_results, indent=2))

--- Analyzing Squats ---

Processing squats video: /content/Squats.mp4
Calibration: Absolute range 47.3° to 179.9°
Ultra-Sensitivity Thresholds: Down < 146.7°, Up > 146.7°
Results: 11 reps counted.

--- Analyzing Pushups ---

Processing pushups video: /content/PushUps.mp4
Calibration: Absolute range 71.8° to 163.3°
Ultra-Sensitivity Thresholds: Down < 144.1°, Up > 144.1°
Results: 12 reps counted.

Analysis complete. Final JSON Result:
{
  "video_id": "3023718e-0b02-4233-be85-4eb19e5dd980",
  "summary": {
    "squats": {
      "total_reps": 11,
      "good_form_reps": 9
    },
    "pushups": {
      "total_reps": 12,
      "good_form_reps": 11
    }
  }
}


## 7. General Pose Visualization for New Videos

This section provides a general pose estimation visualization for your new videos, drawing keypoints and the skeletal structure on each frame and saving the results as new video files.

In [16]:
# Define paths for the new videos
video_path_1 = '/content/PushUps.mp4'
video_path_2 = '/content/Squats.mp4'

print(f"New Video 1 Path: {video_path_1}")
print(f"New Video 2 Path: {video_path_2}")

New Video 1 Path: /content/PushUps.mp4
New Video 2 Path: /content/Squats.mp4


In [17]:
import cv2
import numpy as np

# Real-time EMA for a single scalar. Alpha parameter can be adjusted.
def real_time_ema_scalar(current_value, previous_smoothed_value, alpha=0.2):
    if previous_smoothed_value is None:
        return current_value
    return alpha * current_value + (1 - alpha) * previous_smoothed_value

def calculate_angle(keypoints, p1_idx, p2_idx, p3_idx):
    """
    Calculates the angle (in degrees) between three keypoints.
    p2_idx is the vertex of the angle.
    """
    # Ensure keypoints are valid (x, y, confidence) format and points exist
    if p1_idx >= len(keypoints) or p2_idx >= len(keypoints) or p3_idx >= len(keypoints):
        return None

    # Extract coordinates
    p1 = np.array([keypoints[p1_idx][0], keypoints[p1_idx][1]])
    p2 = np.array([keypoints[p2_idx][0], keypoints[p2_idx][1]])
    p3 = np.array([keypoints[p3_idx][0], keypoints[p3_idx][1]])

    # Calculate vectors
    v1 = p1 - p2
    v2 = p3 - p2

    # Calculate dot product and magnitudes
    dot_product = np.dot(v1, v2)
    magnitude_v1 = np.linalg.norm(v1)
    magnitude_v2 = np.linalg.norm(v2)

    if magnitude_v1 == 0 or magnitude_v2 == 0:
        return None # Avoid division by zero

    # Calculate cosine of the angle
    cosine_angle = dot_product / (magnitude_v1 * magnitude_v2)
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0) # Clip to avoid floating point errors

    # Calculate angle in radians and then convert to degrees
    angle_rad = np.arccos(cosine_angle)
    angle_deg = np.degrees(angle_rad)

    return angle_deg

def visualize_keypoints_on_video(input_video_path, exercise_type, output_video_name='output_video.mp4'):
    """
    Processes a video, detects keypoints using the YOLO model, draws them on frames,
    and saves the annotated video, including a real-time rep counter.

    Args:
        input_video_path (str): Path to the input video file.
        exercise_type (str): 'squats' or 'pushups', for rep counting logic.
        output_video_name (str): Name for the output video file.
    """
    print(f"\nVisualizing keypoints and counting reps for {exercise_type} in: {input_video_path}")

    # --- First Pass: Collect angles for dynamic thresholding ---
    cap_first_pass = cv2.VideoCapture(input_video_path)
    if not cap_first_pass.isOpened():
        print(f"Error: Could not open video file {input_video_path} for first pass")
        return

    all_raw_angles = []
    while True:
        ret, frame = cap_first_pass.read()
        if not ret:
            break
        results = model(frame, verbose=False)
        if results[0].keypoints.data.numel() > 0:
            keypoints = results[0].keypoints.data[0].cpu().numpy()
            angle = None
            if exercise_type == 'squats':
                angle = calculate_angle(keypoints, 11, 13, 15) # Left Hip-Knee-Ankle
            elif exercise_type == 'pushups':
                angle = calculate_angle(keypoints, 5, 7, 9)   # Left Shoulder-Elbow-Wrist
            if angle is not None: # Ensure angle is valid before appending
                all_raw_angles.append(angle)
    cap_first_pass.release()

    if not all_raw_angles:
        print(f"No keypoints or angles detected in {input_video_path} during first pass. Cannot count reps.")
        return

    # --- Dynamic Threshold Calibration (based on successful process_video logic) ---
    min_a = min(all_raw_angles)
    max_a = max(all_raw_angles)
    range_a = max_a - min_a

    if exercise_type == 'pushups':
        # Used 0.21 in process_video for ultra-sensitivity for pushups
        threshold_percentage = 0.21
    else: # squats
        # Used 0.25 in process_video for ultra-sensitivity for squats
        threshold_percentage = 0.25

    # The point at which the angle is considered 'down' or 'up' in the main range
    dynamic_threshold_val = max_a - (range_a * threshold_percentage)

    # Apply a small buffer for entering 'down' state, but 'up' trigger is same as dynamic_threshold_val
    dynamic_down_trigger = dynamic_threshold_val - 1 # Trigger down if angle drops below this
    dynamic_up_trigger = dynamic_threshold_val       # Trigger up if angle rises above this

    print(f"Calibration: Absolute range {min_a:.1f} to {max_a:.1f}")
    print(f"Dynamic Thresholds (Trigger points): Down < {dynamic_down_trigger:.1f}, Up > {dynamic_up_trigger:.1f}")

    # --- Second Pass: Visualization and Rep Counting ---
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {input_video_path} for second pass")
        return

    # Get video properties for VideoWriter
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for .mp4
    out = cv2.VideoWriter(output_video_name, fourcc, fps, (frame_width, frame_height))

    # Initialize rep counting variables
    total_reps = 0
    good_reps = 0
    bad_reps = 0
    rep_state = 0 # State for the rep detection logic (0: Up/Initial, 1: Down)
    deepest_angle_in_rep = None
    current_smoothed_angle = None # For real-time EMA on the angle itself
    ema_alpha = 0.5 # Increased smoothing factor for angles for display/real-time decision to be more reactive

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Perform pose estimation
        results = model(frame, verbose=False)

        # Draw keypoints and skeleton on the frame
        annotated_frame = results[0].plot()

        keypoints = None
        if results[0].keypoints.data.numel() > 0:
            keypoints = results[0].keypoints.data[0].cpu().numpy() # x, y, confidence

        if keypoints is not None:
            current_raw_angle = None
            if exercise_type == 'squats':
                current_raw_angle = calculate_angle(keypoints, 11, 13, 15) # Left Hip-Knee-Ankle
            elif exercise_type == 'pushups':
                current_raw_angle = calculate_angle(keypoints, 5, 7, 9)   # Left Shoulder-Elbow-Wrist

            if current_raw_angle is not None:
                current_smoothed_angle = real_time_ema_scalar(current_raw_angle, current_smoothed_angle, ema_alpha)

                # Rep Counting Logic using dynamic thresholds
                if rep_state == 0: # Up/Initial state
                    # Trigger descent if current angle drops below dynamic_down_trigger
                    if current_smoothed_angle < dynamic_down_trigger:
                        rep_state = 1
                        deepest_angle_in_rep = current_smoothed_angle

                elif rep_state == 1: # Down state
                    # Keep track of the deepest angle achieved during the rep
                    if deepest_angle_in_rep is None or current_smoothed_angle < deepest_angle_in_rep:
                        deepest_angle_in_rep = current_smoothed_angle

                    # Trigger ascent and rep completion if current angle rises above dynamic_up_trigger
                    if current_smoothed_angle > dynamic_up_trigger:
                        total_reps += 1

                        # Form Assessment (using criteria that worked in `process_video` cell 90e0c0ae)
                        is_good = "Bad Form (General)"
                        if deepest_angle_in_rep is not None:
                            if exercise_type == 'squats':
                                is_good = "Good Form" if deepest_angle_in_rep < 110 else "Bad Form (Depth)"
                            else: # Pushups
                                is_good = "Good Form" if deepest_angle_in_rep < 100 else "Bad Form (Depth)"

                        if is_good == "Good Form":
                            good_reps += 1
                        else:
                            bad_reps += 1

                        rep_state = 0
                        deepest_angle_in_rep = None

        # Display rep counter on the frame
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1
        font_thickness = 2

        # Display Total, Good, and Bad Reps
        cv2.putText(annotated_frame, f'Total Reps: {total_reps}', (10, 50), font, font_scale, (255, 255, 255), font_thickness, cv2.LINE_AA)
        cv2.putText(annotated_frame, f'Good Reps: {good_reps}', (10, 100), font, font_scale, (0, 255, 0), font_thickness, cv2.LINE_AA)
        cv2.putText(annotated_frame, f'Bad Reps: {bad_reps}', (10, 150), font, font_scale, (0, 0, 255), font_thickness, cv2.LINE_AA)

        # Write the annotated frame to the output video
        out.write(annotated_frame)

        frame_count += 1
        if frame_count % 100 == 0:
            print(f"Processed {frame_count} frames for {input_video_path}")

    cap.release()
    out.release()
    print(f"Finished visualizing keypoints and reps for {input_video_path}. Output saved as {output_video_name}")

In [18]:
model = YOLO('yolov8s-pose.pt')

# Visualize keypoints for the first new video (now pushups)
visualize_keypoints_on_video(video_path_1, 'pushups', 'annotated_video_1_pushups_reps.mp4')

# Visualize keypoints for the second new video (now squats)
visualize_keypoints_on_video(video_path_2, 'squats', 'annotated_video_2_squats_reps.mp4')


Visualizing keypoints and counting reps for pushups in: /content/PushUps.mp4
Calibration: Absolute range 71.8 to 163.3
Dynamic Thresholds (Trigger points): Down < 143.1, Up > 144.1
Processed 100 frames for /content/PushUps.mp4
Processed 200 frames for /content/PushUps.mp4
Processed 300 frames for /content/PushUps.mp4
Processed 400 frames for /content/PushUps.mp4
Processed 500 frames for /content/PushUps.mp4
Finished visualizing keypoints and reps for /content/PushUps.mp4. Output saved as annotated_video_1_pushups_reps.mp4

Visualizing keypoints and counting reps for squats in: /content/Squats.mp4
Calibration: Absolute range 47.3 to 179.9
Dynamic Thresholds (Trigger points): Down < 145.7, Up > 146.7
Processed 100 frames for /content/Squats.mp4
Processed 200 frames for /content/Squats.mp4
Processed 300 frames for /content/Squats.mp4
Processed 400 frames for /content/Squats.mp4
Processed 500 frames for /content/Squats.mp4
Processed 600 frames for /content/Squats.mp4
Processed 700 frames 